In [8]:
# Importing libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn import metrics


import tensorflow as tf
import keras
from keras import layers

In [9]:
df = pd.read_csv("housing.csv")

In [10]:
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


In [11]:
df.describe()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value
count,20640.000000,20640.000000,20640.000000,20640.000000,20433.000000,20640.000000,20640.000000,20640.000000,20640.000000
mean,-119.569704,35.631861,28.639486,2635.763081,537.870553,1425.476744,499.539680,3.870671,206855.816909
std,2.003532,2.135952,12.585558,2181.615252,421.385070,1132.462122,382.329753,1.899822,115395.615874
min,-124.350000,32.540000,1.000000,2.000000,1.000000,3.000000,1.000000,0.499900,14999.000000
25%,-121.800000,33.930000,18.000000,1447.750000,296.000000,787.000000,280.000000,2.563400,119600.000000
50%,-118.490000,34.260000,29.000000,2127.000000,435.000000,1166.000000,409.000000,3.534800,179700.000000
75%,-118.010000,37.710000,37.000000,3148.000000,647.000000,1725.000000,605.000000,4.743250,264725.000000
max,-114.310000,41.950000,52.000000,39320.000000,6445.000000,35682.000000,6082.000000,15.000100,500001.000000


#### For further analysis I will create a new column "distance_to_nearest_city" using haversine and I will drop lon and lat. Geographical coordinates are not good for machine learning to use, they can create problems. 

In [12]:
# here I will create a new feature "distance_to_nearest_city" based on longitude and latitude
# approximate coordinates of 2 biggerst cities:
sf_coords = (37.7749, -122.4194)  # San Francisco
la_coords = (34.0522, -118.2437)  # Los Angeles

In [13]:
# Use haversine formula (distance in km):

from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c

In [14]:
# Create a column for the shortest distance:

df['distance_to_nearest_city'] = df.apply(
    lambda row: round(min(
        haversine(row['latitude'], row['longitude'], *sf_coords),
        haversine(row['latitude'], row['longitude'], *la_coords)
    ), 2),
    axis=1
)
# ChatGPT helped me to calculate the distance using haversine formula

In [15]:
# now i will drop lon and lat
df = df.drop(columns=['latitude', 'longitude'])
df.head()

,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,distance_to_nearest_city
0,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY,20.33
1,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY,19.91
2,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY,17.84
3,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY,17.06
4,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY,17.06


In [16]:
# "ocean_proximity" is nominal categorical variable, I will use one-hot encoding to convert it to numerical
df['ocean_proximity'].unique()

array(['NEAR BAY', '<1H OCEAN', 'INLAND', 'NEAR OCEAN', 'ISLAND'],
      dtype=object)

In [17]:
# by using encoder we will add 5 new columns, but will remove one -  and the original one.
from sklearn.preprocessing import OneHotEncoder
variables = ['ocean_proximity']

# use encoder
encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
one_hot_encoded = encoder.fit_transform(df[variables]).astype(int)
df = pd.concat([df,one_hot_encoded],axis=1).drop(columns=variables)

In [18]:
df.head()

,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,distance_to_nearest_city,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
0,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,20.33,0,0,0,1,0
1,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,19.91,0,0,0,1,0
2,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,17.84,0,0,0,1,0
3,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,17.06,0,0,0,1,0
4,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,17.06,0,0,0,1,0


In [19]:
dfsds

NameError: name 'dfsds' is not defined

In [ ]:
# to use this dataset in other tools I will create a new csv file. This dataset will be without longitude and latitude.
df.to_csv("housing_modified.csv", index=False)